# Training a Health Classifier

Real GMWI2 doesn't use a hand-picked formula — it trains a **Lasso-
penalized logistic regression** model on thousands of labeled samples,
and evaluates it with **cross-validation**, reporting **balanced
accuracy**. This notebook does exactly that, for real, using
**scikit-learn** (bundled in this browser, no install needed) on the same
teaching dataset — at a much smaller scale than the real 8,069-sample
study, but the identical methodology.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # silence sklearn version-transition noise; nothing here affects the math

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import balanced_accuracy_score

df = pd.read_csv("toy_species_abundance.csv")
species = [c for c in df.columns if c not in ("sample_id", "health_status")]

X = (df[species] > 0).astype(int).values   # presence/absence, same as notebook 03
y = (df["health_status"] == "healthy").astype(int).values  # 1 = healthy, 0 = non_healthy
print(X.shape, y.shape, "->", y.sum(), "healthy /", len(y) - y.sum(), "non_healthy")

## 2. Why Lasso specifically?

**Lasso** (L1-penalized) logistic regression doesn't just fit weights —
it actively pushes unhelpful weights to *exactly* zero. With 12 species
and only 80 samples, that matters: it's a built-in way to ask "which of
these species actually carries signal?" instead of forcing every feature
to contribute something. The real GMWI2 does this over hundreds of
candidate species; the shrinkage is what turns "hundreds of species" into
a short, interpretable list of ones that actually matter.

In [ ]:
model = LogisticRegression(penalty="l1", solver="liblinear", C=0.5)
model.fit(X, y)

weights = pd.Series(model.coef_[0], index=species).sort_values()
print(weights.round(3))

### EXPLAIN #1
*Look at the sign of each species' weight. Do the six species you'd
expect to be "health-prevalent" (from notebook 04) get positive weights,
and the "health-scarce" ones negative? Are any weights exactly `0.0` —
and if so, what does Lasso seem to be saying about those species?*

> your answer here

## 3. Cross-validation: don't trust a score you trained on

Evaluating a model on the same data it learned from overestimates how
good it really is. **K-fold cross-validation** splits the data into K
groups, trains on K-1 of them, and tests on the held-out group — repeated
K times so every sample gets tested exactly once, by a model that never
saw it during training. This is the same principle behind GMWI2's real
evaluation.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(model, X, y, cv=cv, scoring="balanced_accuracy")
print("fold scores:", scores.round(3))
print(f"mean balanced accuracy: {scores.mean():.3f}")

**Balanced accuracy** averages the accuracy on each class separately —
important here since it's exactly the metric the real paper reports:
GMWI2 achieves roughly **80% balanced accuracy** overall, and over **90%**
on its highest-confidence predictions. Your toy model, on 80 much simpler
synthetic samples, should land somewhere in a broadly similar range — not
because the numbers are meant to match exactly, but because the same
evaluation methodology tends to produce moderately-imperfect, not
perfect, scores on real biological data.

### 🔧 YOUR TURN #2
Change `n_splits=5` to `n_splits=10`, or change `C=0.5` to `C=0.1` (a
stronger penalty, expect more zero weights) in the model above and
re-run. Does mean balanced accuracy change much? Real papers usually try
several settings like this before reporting one.

In [ ]:
# Your code here

## Done — you just did what the paper did

Different data, much smaller scale, but the same core method: presence/
absence features, L1-penalized logistic regression, cross-validated
balanced accuracy. Notebook 06 shows what running the *actual* trained
GMWI2 model looks like in a real lab.

**Next:** `06_running_gmwi2_for_real.ipynb`.